# Swiss Legal Citation Retrieval: Recall-Preserving Candidate Funnel

This Colab notebook builds a controlled, multi-stage candidate generation system for Swiss legal citation retrieval.

The central rule is strict: **all final candidates must come from the source `citation` columns of `laws_de.csv` or `court_considerations.csv`**. Regex, LLM planner output, embeddings, and rerankers may only choose routes and filters; they may never invent final citation IDs.

The notebook handles laws and court considerations separately, audits recall after every narrowing stage, and only then merges candidates into a 2k-10k recall-first pool.

## 0. Colab Setup

Expected Drive layout:

```text
/content/drive/MyDrive/swiss_law/data
/content/drive/MyDrive/swiss_law/data_insights
/content/drive/MyDrive/swiss_law/artifacts
```

The heavy model cells are optional. You can first run the deterministic funnel and recall audit, then enable the Qwen planner/reranker cells.

In [1]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

LOCAL_BASE_DIR = Path(os.environ.get("SWISS_LAW_BASE_DIR", r"E:\swiss_citation_extraction"))
BASE_DIR = Path("/content/drive/MyDrive/swiss_law") if IN_COLAB else LOCAL_BASE_DIR
DATA_DIR = BASE_DIR / "data"
INSIGHTS_DIR = BASE_DIR / "data_insights"
ART_DIR = BASE_DIR / "artifacts"
ART_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_DIR    :", BASE_DIR)
print("DATA_DIR    :", DATA_DIR)
print("INSIGHTS_DIR:", INSIGHTS_DIR)
print("ART_DIR     :", ART_DIR)

Mounted at /content/drive
BASE_DIR    : /content/drive/MyDrive/swiss_law
DATA_DIR    : /content/drive/MyDrive/swiss_law/data
INSIGHTS_DIR: /content/drive/MyDrive/swiss_law/data_insights
ART_DIR     : /content/drive/MyDrive/swiss_law/artifacts


In [2]:
if IN_COLAB:
    import subprocess
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-U",
        "duckdb",
        "polars",
        "pandas",
        "pyarrow",
        "orjson",
        "ijson",
        "regex",
        "rapidfuzz",
        "transformers",
        "accelerate",
        "bitsandbytes",
        "sentence-transformers",
        "faiss-cpu",
    ])

In [3]:
import csv
import gc
import math
import re
import time
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Iterable

import duckdb
import pandas as pd
import regex as regex
from rapidfuzz import fuzz

try:
    import orjson

    def loads_json(value: str | bytes) -> Any:
        return orjson.loads(value)

    def dumps_json(value: Any) -> str:
        return orjson.dumps(value, option=orjson.OPT_INDENT_2 | orjson.OPT_SORT_KEYS).decode("utf-8")

except Exception:
    import json

    def loads_json(value: str | bytes) -> Any:
        if isinstance(value, bytes):
            value = value.decode("utf-8")
        return json.loads(value)

    def dumps_json(value: Any) -> str:
        return json.dumps(value, indent=2, sort_keys=True, ensure_ascii=False)

FILES = {
    "laws_csv": DATA_DIR / "laws_de.csv",
    "court_csv": DATA_DIR / "court_considerations.csv",
    "train_csv": DATA_DIR / "train.csv",
    "val_csv": DATA_DIR / "val.csv",
    "test_csv": DATA_DIR / "test.csv",
    "laws_classified": INSIGHTS_DIR / "laws_de_classified_citations.jsonl",
    "court_classified": INSIGHTS_DIR / "court_considerations_classified_citations.jsonl",
    "laws_links": INSIGHTS_DIR / "laws_de_links.json",
    "court_links": INSIGHTS_DIR / "court_considerations_links.json",
}

for name, path in FILES.items():
    print(f"{name:18s}", "OK" if path.exists() else "MISSING", path)

laws_csv           OK /content/drive/MyDrive/swiss_law/data/laws_de.csv
court_csv          OK /content/drive/MyDrive/swiss_law/data/court_considerations.csv
train_csv          OK /content/drive/MyDrive/swiss_law/data/train.csv
val_csv            OK /content/drive/MyDrive/swiss_law/data/val.csv
test_csv           OK /content/drive/MyDrive/swiss_law/data/test.csv
laws_classified    OK /content/drive/MyDrive/swiss_law/data_insights/laws_de_classified_citations.jsonl
court_classified   OK /content/drive/MyDrive/swiss_law/data_insights/court_considerations_classified_citations.jsonl
laws_links         OK /content/drive/MyDrive/swiss_law/data_insights/laws_de_links.json
court_links        OK /content/drive/MyDrive/swiss_law/data_insights/court_considerations_links.json


## 1. Configuration

Set `RUN_SPLIT = "val"` while developing. For test inference, use `RUN_SPLIT = "test"`; the recall audit will still produce candidate counts, but no gold recall.

`RECALL_GUARD_ON_VAL` is a development safety switch. When a validation stage drops recall below its gate, the notebook keeps the previous wider candidate set and records the failed stage. This prevents one bad planner choice from silently destroying the candidate pool.

In [4]:
RUN_SPLIT = "val"  # "val", "test", or "train"

BUILD_TEXT_TABLES = True      # Builds DuckDB tables with citation text. Slow but useful for reranking.
BUILD_EDGE_TABLES = True      # Streams links JSON into citation_edges for statute-citing court routes.
FORCE_REBUILD_DB = False      # Set True after changing parsing/index logic.

USE_PLANNER_LLM = True        # Planner LLM produces structural choices. Falls back to heuristic on error.
USE_RERANKER = False          # Optional later stage.

PLANNER_MODEL_NAME = "Qwen/Qwen3-32B"
RERANKER_MODEL_NAME = "Qwen/Qwen3-Reranker-8B"
EMBEDDING_MODEL_NAME = "Qwen/Qwen3-Embedding-8B"

# Recall guards are diagnostic only. They never look at gold to decide what to keep.
EARLY_STAGE_MIN_RECALL = 0.98
MID_STAGE_MIN_RECALL = 0.95
FINAL_MIN_MEAN_RECALL = 0.85
FINAL_MIN_QUERY_RECALL = 0.75

# Hard per-side cap. Both law and court funnels stop at 2,000 candidates each;
# merged total is capped at 4,000.
MIN_TOTAL_CANDIDATES = 2_000
DEFAULT_TOTAL_CANDIDATES = 4_000
MAX_TOTAL_CANDIDATES = 4_000
DIAGNOSTIC_MAX_CANDIDATES = 4_000

DEFAULT_LAW_BUDGET = 2_000
DEFAULT_COURT_BUDGET = 2_000

# Soft-rank funnel knobs (used by run_*_funnel_soft below in section 9b)
SOFT_FUNNEL_ENABLED = True
SOFT_LAW_TARGET = 2_000
SOFT_COURT_TARGET = 2_000
# Probe multipliers were used to grow K against gold on val. That is leakage.
# Keep a single step so the cap is always honored on every split.
SOFT_PROBE_STEPS = [1.0]

# LEAKAGE WARNING. The gold-bank path below uses data_insights/*_gold_citations.json
# as the candidate set. For val with ["val"] this trivially yields recall 1.0
# (the answer key IS the candidate list). For test with ["train", "val"] it
# still relies on labels that overlap with the eval target. It is kept only as
# a diagnostic for the candidate plumbing and is OFF by default.
USE_GOLD_BANK_CANDIDATES = False
GOLD_BANK_SPLITS_FOR_VAL = ["val"]
GOLD_BANK_SPLITS_FOR_TEST = ["train", "val"]
GOLD_BANK_LAW_CAP = 2_000
GOLD_BANK_COURT_CAP = 2_000

DB_PATH = ART_DIR / "recall_funnel.duckdb"
print("DB_PATH:", DB_PATH)
print("USE_PLANNER_LLM         :", USE_PLANNER_LLM)
print("USE_GOLD_BANK_CANDIDATES:", USE_GOLD_BANK_CANDIDATES, "(leakage path; off by default)")
print("LAW/COURT budget        :", DEFAULT_LAW_BUDGET, "/", DEFAULT_COURT_BUDGET)


DB_PATH: /content/drive/MyDrive/swiss_law/artifacts/recall_funnel.duckdb
USE_PLANNER_LLM         : True
USE_GOLD_BANK_CANDIDATES: False (leakage path; off by default)
LAW/COURT budget        : 2000 / 2000


## 2. Regex And Citation Normalization Layer

The regex layer is intentionally broad. It is used to detect query mentions and generate variant expansions. Final candidates still come only from the source citation inventory.

In [5]:
WS_RE = re.compile(r"\s+")

LAW_REF_RE = regex.compile(
    r'''
    (?P<marker>Art\.|Artikel)\s+
    (?P<article>\d+[a-zA-Z]*(?:\s*(?:bis|ter|quater))?)
    (?P<continuation>\s*(?:f\.|ff\.))?
    (?P<units>
        (?:
            \s+
            (?:
                (?:Abs\.|Absatz|Absatze|Absätze)\s+\d+[a-zA-Z]*(?:bis)?
                |(?:Bst\.|lit\.|Buchstabe)\s+[a-zA-Z]
                |(?:Ziff\.|Ziffer|Ziffern|Nr\.)\s+\d+[a-zA-Z]*
                |(?:Satz)\s+\d+
                |(?:Unterabs\.)\s+\d+
            )
        )*
    )
    (?:\s+(?P<law_code>[A-ZÄÖÜ][A-Za-zÄÖÜäöü0-9./-]{1,40}))?
    ''',
    regex.VERBOSE,
)

BGE_RE = regex.compile(
    r"\bBGE\s+(?P<volume>\d{2,3})\s+(?P<division>[IVX]{1,5})\s+(?P<page>\d{1,4})(?:\s+E\.?\s*(?P<pinpoint>[A-Za-z0-9.:-]+))?",
    regex.IGNORECASE,
)

MODERN_DOCKET_RE = regex.compile(
    r"\b(?P<docket>\d{1,2}[A-Z]{1,2}[_\.]\d{1,5}/\d{4})(?:\s+(?P<date>\d{2}\.\d{2}\.\d{4}))?(?:\s+E\.?\s*(?P<consideration>[A-Za-z0-9.:-]+))?",
    regex.IGNORECASE,
)

LEGACY_DOCKET_RE = regex.compile(
    r"\b(?P<docket>[A-Z]\s+\d{1,5}/\d{2})(?:\s+(?P<date>\d{2}\.\d{2}\.\d{4}))?(?:\s+E\.?\s*(?P<consideration>[A-Za-z0-9.:-]+))?",
    regex.IGNORECASE,
)

UNKNOWN_DOCKET_LIKE_RE = regex.compile(
    r"\b(?P<docket>(?:\d{1,2}[A-Z]{1,2}[_\.]\d{1,5}/\d{4}|[A-Z]\s+\d{1,5}/\d{2}))",
    regex.IGNORECASE,
)

DATE_TRIGGER_RE = regex.compile(
    r"\b(after|before|since|until|between|recent|newer|older|from\s+\d{4}|post[-\s]?\d{4}|pre[-\s]?\d{4}|in force|entered into force|law in force)\b",
    regex.IGNORECASE,
)


def squash_ws(value: str | None) -> str:
    return WS_RE.sub(" ", value or "").strip()


def unit_chain_from_units(units: list[dict] | None) -> str:
    units = units or []
    return ">".join(str(u.get("category") or "").strip() for u in units if u.get("category"))


def unit_values_from_units(units: list[dict] | None) -> str:
    units = units or []
    values = []
    for unit in units:
        category = unit.get("category") or ""
        value = unit.get("value") or ""
        if category or value:
            values.append(f"{category}:{value}")
    return "|".join(values)


def parse_docket_prefix(docket: str | None) -> str | None:
    if not docket:
        return None
    docket = squash_ws(docket)
    if "_" in docket:
        return docket.split("_", 1)[0]
    if "." in docket and "/" in docket:
        return docket.split(".", 1)[0]
    if " " in docket:
        return docket.split(" ", 1)[0]
    return None


def normalize_court_base(citation: str, pattern: str, segments: dict) -> str:
    if pattern == "court_bge":
        return f"BGE {segments.get('volume')} {segments.get('division')} {segments.get('page')}"
    if pattern == "court_case":
        return segments.get("docket") or citation
    return citation


def extract_query_mentions(query: str) -> dict[str, list[dict]]:
    mentions = {"laws": [], "bge": [], "dockets": [], "dates": []}
    for m in LAW_REF_RE.finditer(query):
        mentions["laws"].append({k: squash_ws(v) if v else None for k, v in m.groupdict().items()})
    for m in BGE_RE.finditer(query):
        gd = {k: squash_ws(v) if v else None for k, v in m.groupdict().items()}
        gd["base"] = f"BGE {gd['volume']} {gd['division']} {gd['page']}"
        mentions["bge"].append(gd)
    for rx in (MODERN_DOCKET_RE, LEGACY_DOCKET_RE):
        for m in rx.finditer(query):
            gd = {k: squash_ws(v) if v else None for k, v in m.groupdict().items()}
            gd["prefix"] = parse_docket_prefix(gd.get("docket"))
            mentions["dockets"].append(gd)
    mentions["time_mode"] = "hard_year_range" if DATE_TRIGGER_RE.search(query) else "no_time_filter"
    return mentions

## 3. Build Compact DuckDB Indexes

This creates source-only citation segment tables and optional text/edge tables. It also exports parquet copies so later reruns start quickly.

In [6]:
def segment_record(dataset: str, row: dict) -> tuple:
    citation = row["citation"]
    pattern = row.get("pattern") or "unknown"
    family = row.get("family") or "unknown"
    subfamily = row.get("subfamily") or "unknown"
    origin_mask = int(row.get("origin_mask") or 0)
    seg = row.get("segments") or {}

    article = seg.get("article") or seg.get("article_number")
    section = seg.get("section") or seg.get("section_number")
    law_code = seg.get("law_code")
    law_code_family = seg.get("law_code_family")
    law_code_resolution = seg.get("law_code_resolution")
    article_continuation = seg.get("article_continuation") or seg.get("article_suffix_or_range")
    units = seg.get("units") or []
    unit_chain = unit_chain_from_units(units)
    unit_values = unit_values_from_units(units)
    unit_depth = len(units)

    docket = seg.get("docket")
    docket_prefix = parse_docket_prefix(docket)
    legal_area_code = seg.get("legal_area_code")
    court_chamber = seg.get("court_chamber")
    separator_style = seg.get("separator_style")
    decision_year = seg.get("decision_year")
    decision_date = seg.get("decision_date")
    consideration = seg.get("consideration") or seg.get("pinpoint")
    bge_volume = seg.get("volume")
    bge_division = seg.get("division")
    bge_page = seg.get("page")
    court_base = normalize_court_base(citation, pattern, seg)
    raw = seg.get("raw") or citation

    if pattern == "unknown":
        m = UNKNOWN_DOCKET_LIKE_RE.search(citation)
        if m:
            docket = docket or squash_ws(m.group("docket"))
            docket_prefix = docket_prefix or parse_docket_prefix(docket)
            court_base = docket

    return (
        dataset,
        citation,
        origin_mask,
        family,
        subfamily,
        pattern,
        article,
        section,
        law_code,
        law_code_family,
        law_code_resolution,
        article_continuation,
        unit_chain,
        unit_values,
        unit_depth,
        docket,
        docket_prefix,
        legal_area_code,
        court_chamber,
        separator_style,
        decision_year,
        decision_date,
        consideration,
        bge_volume,
        bge_division,
        bge_page,
        court_base,
        raw,
    )


SEGMENT_SCHEMA = '''
    dataset TEXT,
    citation TEXT,
    origin_mask INTEGER,
    family TEXT,
    subfamily TEXT,
    pattern TEXT,
    article TEXT,
    section TEXT,
    law_code TEXT,
    law_code_family TEXT,
    law_code_resolution TEXT,
    article_continuation TEXT,
    unit_chain TEXT,
    unit_values TEXT,
    unit_depth INTEGER,
    docket TEXT,
    docket_prefix TEXT,
    legal_area_code TEXT,
    court_chamber TEXT,
    separator_style TEXT,
    decision_year TEXT,
    decision_date TEXT,
    consideration TEXT,
    bge_volume TEXT,
    bge_division TEXT,
    bge_page TEXT,
    court_base TEXT,
    raw TEXT
'''


def create_segment_table(con: duckdb.DuckDBPyConnection, table: str, path: Path, dataset: str, batch_size: int = 100_000) -> None:
    con.execute(f"DROP TABLE IF EXISTS {table}")
    con.execute(f"CREATE TABLE {table} ({SEGMENT_SCHEMA})")
    insert_sql = f"INSERT INTO {table} VALUES ({','.join(['?'] * 28)})"

    batch = []
    kept = 0
    seen = 0
    t0 = time.time()
    with path.open("rb") as f:
        for raw_line in f:
            if not raw_line.strip():
                continue
            seen += 1
            row = loads_json(raw_line)
            origin_mask = int(row.get("origin_mask") or 0)
            if not (origin_mask & 1):
                continue
            batch.append(segment_record(dataset, row))
            kept += 1
            if len(batch) >= batch_size:
                con.executemany(insert_sql, batch)
                batch.clear()
                print(f"{table}: inserted {kept:,} source rows after {seen:,} jsonl rows ({time.time() - t0:.1f}s)")
    if batch:
        con.executemany(insert_sql, batch)
    print(f"{table}: done. source rows={kept:,}, jsonl rows scanned={seen:,}")


def create_text_tables(con: duckdb.DuckDBPyConnection) -> None:
    con.execute("DROP TABLE IF EXISTS law_citations")
    con.execute(
        f'''
        CREATE TABLE law_citations AS
        SELECT citation, title, text
        FROM read_csv_auto('{FILES['laws_csv'].as_posix()}', header=true, all_varchar=true, ignore_errors=true)
        '''
    )
    con.execute("DROP TABLE IF EXISTS court_considerations")
    con.execute(
        f'''
        CREATE TABLE court_considerations AS
        SELECT citation, text
        FROM read_csv_auto('{FILES['court_csv'].as_posix()}', header=true, all_varchar=true, ignore_errors=true)
        '''
    )


def create_edge_table(con: duckdb.DuckDBPyConnection, batch_size: int = 250_000) -> None:
    import ijson

    con.execute("DROP TABLE IF EXISTS citation_edges")
    con.execute("CREATE TABLE citation_edges (dataset TEXT, source TEXT, target TEXT)")
    insert_sql = "INSERT INTO citation_edges VALUES (?, ?, ?)"

    for dataset, path in [("laws_de", FILES["laws_links"]), ("court_considerations", FILES["court_links"])]:
        if not path.exists():
            print("Missing links file:", path)
            continue
        batch = []
        n_edges = 0
        t0 = time.time()
        with path.open("rb") as f:
            for item in ijson.items(f, "source_to_references.item"):
                source = item.get("source")
                for target in item.get("references", []) or []:
                    batch.append((dataset, source, target))
                    n_edges += 1
                    if len(batch) >= batch_size:
                        con.executemany(insert_sql, batch)
                        batch.clear()
                        print(f"{dataset}: inserted {n_edges:,} edges ({time.time() - t0:.1f}s)")
        if batch:
            con.executemany(insert_sql, batch)
        print(f"{dataset}: edge load done, edges={n_edges:,}")

    con.execute("CREATE INDEX IF NOT EXISTS idx_edges_target ON citation_edges(target)")
    con.execute("CREATE INDEX IF NOT EXISTS idx_edges_source ON citation_edges(source)")


def build_or_load_db() -> duckdb.DuckDBPyConnection:
    if FORCE_REBUILD_DB and DB_PATH.exists():
        DB_PATH.unlink()

    con = duckdb.connect(str(DB_PATH))
    existing_tables = {r[0] for r in con.execute("SHOW TABLES").fetchall()}
    need_segments = not {"law_segments", "court_segments"}.issubset(existing_tables)

    if need_segments:
        create_segment_table(con, "law_segments", FILES["laws_classified"], "laws_de")
        create_segment_table(con, "court_segments", FILES["court_classified"], "court_considerations")
        con.execute("CREATE INDEX IF NOT EXISTS idx_law_citation ON law_segments(citation)")
        con.execute("CREATE INDEX IF NOT EXISTS idx_law_code_article ON law_segments(law_code, article)")
        con.execute("CREATE INDEX IF NOT EXISTS idx_court_citation ON court_segments(citation)")
        con.execute("CREATE INDEX IF NOT EXISTS idx_court_base ON court_segments(court_base)")
        con.execute("CREATE INDEX IF NOT EXISTS idx_court_prefix ON court_segments(docket_prefix)")

    existing_tables = {r[0] for r in con.execute("SHOW TABLES").fetchall()}
    if BUILD_TEXT_TABLES and not {"law_citations", "court_considerations"}.issubset(existing_tables):
        create_text_tables(con)

    existing_tables = {r[0] for r in con.execute("SHOW TABLES").fetchall()}
    if BUILD_EDGE_TABLES and "citation_edges" not in existing_tables:
        create_edge_table(con)

    con.execute(
        '''
        CREATE OR REPLACE TABLE court_decisions AS
        SELECT
            court_base,
            any_value(pattern) AS dominant_pattern,
            any_value(subfamily) AS subfamily,
            any_value(docket_prefix) AS docket_prefix,
            any_value(legal_area_code) AS legal_area_code,
            any_value(court_chamber) AS court_chamber,
            any_value(separator_style) AS separator_style,
            any_value(decision_year) AS decision_year,
            any_value(decision_date) AS decision_date,
            any_value(bge_volume) AS bge_volume,
            any_value(bge_division) AS bge_division,
            any_value(bge_page) AS bge_page,
            count(*) AS consideration_count
        FROM court_segments
        GROUP BY court_base
        '''
    )
    return con


con = build_or_load_db()
print(con.execute("SHOW TABLES").fetchdf())
print("law source rows  :", con.execute("SELECT count(*) FROM law_segments").fetchone()[0])
print("court source rows:", con.execute("SELECT count(*) FROM court_segments").fetchone()[0])

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

                   name
0        citation_edges
1  court_considerations
2       court_decisions
3        court_segments
4         law_citations
5          law_segments
law source rows  : 175933
court source rows: 1516398


## 4. Source Inventory And Gold Labels

These sets define valid predictions. Text-reference-only citations are never valid final candidates.

In [7]:
LAW_SOURCE = set(con.execute("SELECT citation FROM law_segments").fetchdf()["citation"])
COURT_SOURCE = set(con.execute("SELECT citation FROM court_segments").fetchdf()["citation"])
ALL_SOURCE = LAW_SOURCE | COURT_SOURCE

print(f"LAW_SOURCE   : {len(LAW_SOURCE):,}")
print(f"COURT_SOURCE : {len(COURT_SOURCE):,}")
print(f"ALL_SOURCE   : {len(ALL_SOURCE):,}")


def split_gold(value: str | float | None) -> list[str]:
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return []
    return [x.strip() for x in str(value).split(";") if x.strip()]


def load_queries(split: str) -> pd.DataFrame:
    path = FILES[f"{split}_csv"]
    df = pd.read_csv(path)
    if "gold_citations" not in df.columns:
        df["gold_citations"] = ""
    df["gold_list"] = df["gold_citations"].apply(split_gold)
    df["gold_set"] = df["gold_list"].apply(set)
    return df


queries = load_queries(RUN_SPLIT)
print("queries:", len(queries), "split:", RUN_SPLIT)
queries.head(2)

LAW_SOURCE   : 175,933
COURT_SOURCE : 1,516,398
ALL_SOURCE   : 1,692,331
queries: 10 split: val


,query_id,query,gold_citations,gold_list,gold_set
0,val_001,May a court lawfully order a three‑month exten...,Art. 221 Abs. 1 StPO;Art. 140 Abs. 1 StGB;Art....,"[Art. 221 Abs. 1 StPO, Art. 140 Abs. 1 StGB, A...","{Art. 100 Abs. 1 BGG, BGE 143 IV 168 E. 5.1, B..."
1,val_002,A claimant holding a national vocational diplo...,Art. 8 Abs. 1 ATSG;Art. 8 Abs. 1 IVG;Art. 17 A...,"[Art. 8 Abs. 1 ATSG, Art. 8 Abs. 1 IVG, Art. 1...","{Art. 4 Abs. 1 IVG, Art. 100 Abs. 1 BGG, BGE 1..."


## 5. Choice Card Generation

Choice cards are generated from actual corpus distributions. The LLM planner may only choose IDs from these cards.

In [8]:
def table_to_records(df: pd.DataFrame) -> list[dict]:
    return loads_json(df.to_json(orient="records"))


def make_choice_cards(con: duckdb.DuckDBPyConnection) -> dict:
    law_codes = con.execute(
        '''
        SELECT
            coalesce(law_code, 'NO_CODE') AS id,
            count(*) AS source_count,
            min(article) AS min_article,
            max(article) AS max_article,
            max(unit_depth) AS max_unit_depth,
            any_value(law_code_family) AS law_code_family
        FROM law_segments
        GROUP BY 1
        ORDER BY source_count DESC
        '''
    ).fetchdf()

    law_titles = pd.DataFrame()
    if "law_citations" in {r[0] for r in con.execute("SHOW TABLES").fetchall()}:
        law_titles = con.execute(
            '''
            SELECT s.law_code, any_value(l.title) AS title
            FROM law_segments s
            JOIN law_citations l USING (citation)
            WHERE s.law_code IS NOT NULL
            GROUP BY s.law_code
            '''
        ).fetchdf()
    if not law_titles.empty:
        law_codes = law_codes.merge(law_titles, how="left", left_on="id", right_on="law_code").drop(columns=["law_code"])
    else:
        law_codes["title"] = None

    law_unit_chains = con.execute(
        '''
        SELECT coalesce(nullif(unit_chain, ''), 'article_only') AS id, count(*) AS source_count
        FROM law_segments
        GROUP BY 1
        ORDER BY source_count DESC
        '''
    ).fetchdf()

    court_families = con.execute(
        '''
        SELECT pattern AS id, subfamily, count(*) AS source_count
        FROM court_segments
        GROUP BY pattern, subfamily
        ORDER BY source_count DESC
        '''
    ).fetchdf()

    bge_divisions = con.execute(
        '''
        SELECT bge_division AS id, count(*) AS source_count
        FROM court_segments
        WHERE pattern = 'court_bge' AND bge_division IS NOT NULL
        GROUP BY 1
        ORDER BY source_count DESC
        '''
    ).fetchdf()
    bge_meanings = {
        "I": "constitutional and public-law leading decisions",
        "II": "public, administrative, tax, migration, and regulatory leading decisions",
        "III": "civil-law leading decisions",
        "IV": "criminal-law and criminal-procedure leading decisions",
        "V": "social-insurance leading decisions",
    }
    bge_divisions["meaning"] = bge_divisions["id"].map(bge_meanings).fillna("BGE division observed in corpus")

    docket_prefixes = con.execute(
        '''
        SELECT docket_prefix AS id, count(*) AS source_count, any_value(legal_area_code) AS legal_area_code
        FROM court_segments
        WHERE docket_prefix IS NOT NULL
        GROUP BY 1
        ORDER BY source_count DESC
        '''
    ).fetchdf()

    route_choices = [
        {"id": "explicit_exact_and_variants", "meaning": "Use exact query mentions and source-inventory variants."},
        {"id": "law_code_route", "meaning": "Filter laws by selected law codes and companion procedural codes."},
        {"id": "article_group_route", "meaning": "Expand selected article bases to all valid granular source rows."},
        {"id": "bge_base_route", "meaning": "Expand selected BGE bases to all source considerations."},
        {"id": "docket_base_route", "meaning": "Expand selected docket bases to all source considerations."},
        {"id": "statute_citing_cases", "meaning": "Add court considerations that cite selected law candidates."},
        {"id": "same_decision_neighbors", "meaning": "Keep neighboring considerations from selected decisions."},
        {"id": "regex_unknown_safety", "meaning": "Include malformed or unknown source rows matching selected docket-like families."},
        {"id": "dense_safety", "meaning": "Optional weak dense/lexical safety candidates; never primary."},
    ]

    time_choices = [
        {"id": "no_time_filter", "meaning": "Default. Query dates are treated as facts and do not filter precedent years."},
        {"id": "soft_recency", "meaning": "Boost newer decisions but do not hard filter."},
        {"id": "hard_year_range", "meaning": "Hard filter by year only when the query explicitly asks for a time period."},
        {"id": "exact_decision_date", "meaning": "Use only when an exact court decision date is explicitly requested."},
    ]

    cards = {
        "law_codes": table_to_records(law_codes),
        "law_unit_chains": table_to_records(law_unit_chains),
        "court_families": table_to_records(court_families),
        "bge_divisions": table_to_records(bge_divisions),
        "docket_prefixes": table_to_records(docket_prefixes),
        "route_choices": route_choices,
        "time_choices": time_choices,
    }
    return cards


choice_cards = make_choice_cards(con)
(ART_DIR / "choice_cards.json").write_text(dumps_json(choice_cards), encoding="utf-8")
print("Saved:", ART_DIR / "choice_cards.json")
for key, values in choice_cards.items():
    print(f"{key:16s}", len(values))
print("Top law codes:", choice_cards["law_codes"][:15])
print("Top docket prefixes:", choice_cards["docket_prefixes"][:15])

Saved: /content/drive/MyDrive/swiss_law/artifacts/choice_cards.json
law_codes        2061
law_unit_chains  2
court_families   3
bge_divisions    0
docket_prefixes  44
route_choices    9
time_choices     4
Top law codes: [{'id': 'NO_CODE', 'source_count': 4419, 'min_article': '1', 'max_article': '9a', 'max_unit_depth': 1, 'law_code_family': 'unresolved_law_code', 'title': None}, {'id': 'ZGB', 'source_count': 2383, 'min_article': '1', 'max_article': '99', 'max_unit_depth': 1, 'law_code_family': 'uppercase_abbreviation', 'title': 'Schweizerisches Zivilgesetzbuch vom 10. Dezember 1907 - A.  Anwendung des Rechts'}, {'id': 'StPO', 'source_count': 1306, 'min_article': '1', 'max_article': '99', 'max_unit_depth': 1, 'law_code_family': 'mixedcase_abbreviation', 'title': 'Schweizerische Strafprozessordnung vom 5. Oktober 2007 (Strafprozessordnung, StPO) - 1. Kapitel:  Geltungsbereich und Ausübung der Strafrechtspflege'}, {'id': 'StGB', 'source_count': 1238, 'min_article': '1', 'max_article': '99'

## 6. Planner

The planner chooses controlled switches. It does **not** output candidate citations.

If `USE_PLANNER_LLM = False`, the notebook uses a broad deterministic fallback so the funnel and recall audit can run immediately.

In [9]:
COMMON_SAFETY_LAW_CODES = {
    "BGG", "BV", "StBOG", "ZGB", "OR", "CO", "StPO", "CPP", "StGB", "CP", "ZPO", "CPC",
    "ATSG", "IVG", "KVG", "UVG", "SchKG", "IPRG", "DBG", "AIG"
}

PREFIX_HINTS = {
    "detention": ["1B", "7B"],
    "custody": ["1B", "7B"],
    "collusion": ["1B", "7B"],
    "criminal": ["1B", "6B", "7B"],
    "offence": ["6B", "1B"],
    "offense": ["6B", "1B"],
    "insurance": ["8C", "9C"],
    "invalidity": ["8C", "9C"],
    "employment": ["8C", "9C", "4A"],
    "contract": ["4A"],
    "company": ["4A"],
    "inheritance": ["5A"],
    "family": ["5A"],
    "civil": ["4A", "5A"],
}

LAW_HINTS = {
    "detention": ["StPO", "StGB", "BGG", "BV", "StBOG"],
    "collusion": ["StPO", "StGB", "BGG", "BV", "StBOG"],
    "criminal": ["StPO", "StGB", "BGG", "BV"],
    "insurance": ["ATSG", "IVG", "KVG", "UVG", "BGG"],
    "invalidity": ["ATSG", "IVG", "BGG"],
    "vocational": ["ATSG", "IVG", "BGG"],
    "contract": ["OR", "CO", "ZGB", "BGG"],
    "company": ["OR", "CO", "ZGB", "BGG"],
    "inheritance": ["ZGB", "BGG"],
    "family": ["ZGB", "BGG"],
    "civil": ["ZGB", "OR", "CO", "ZPO", "BGG"],
}


def allowed_ids(cards: list[dict]) -> set[str]:
    return {str(x.get("id")) for x in cards if x.get("id") is not None}


def heuristic_plan(query: str, cards: dict) -> dict:
    q = query.lower()
    mentions = extract_query_mentions(query)
    law_codes = set()
    for m in mentions["laws"]:
        if m.get("law_code"):
            law_codes.add(m["law_code"])
    for key, codes in LAW_HINTS.items():
        if key in q:
            law_codes.update(codes)
    if not law_codes:
        law_codes.update(COMMON_SAFETY_LAW_CODES)
    else:
        law_codes.update({"BGG", "BV", "StBOG"})

    allowed_law_codes = allowed_ids(cards["law_codes"])
    law_codes = sorted(c for c in law_codes if c in allowed_law_codes or c == "NO_CODE")

    prefixes = set(m.get("prefix") for m in mentions["dockets"] if m.get("prefix"))
    for key, vals in PREFIX_HINTS.items():
        if key in q:
            prefixes.update(vals)
    allowed_prefixes = allowed_ids(cards["docket_prefixes"])
    prefixes = sorted(p for p in prefixes if p in allowed_prefixes)

    bge_divisions = set()
    if any(k in q for k in ["criminal", "detention", "offence", "offense"]):
        bge_divisions.update(["I", "IV"])
    if any(k in q for k in ["civil", "contract", "inheritance", "family", "company"]):
        bge_divisions.add("III")
    if any(k in q for k in ["insurance", "invalidity", "benefits"]):
        bge_divisions.add("V")
    if not bge_divisions:
        bge_divisions.update(["I", "III", "IV", "V"])

    return {
        "law_codes": law_codes,
        "law_unit_chains": [],
        "court_families": ["court_bge", "court_case", "unknown"],
        "bge_divisions": sorted(bge_divisions),
        "docket_prefixes": prefixes,
        "routes": [
            "explicit_exact_and_variants",
            "law_code_route",
            "article_group_route",
            "bge_base_route",
            "docket_base_route",
            "statute_citing_cases",
            "same_decision_neighbors",
            "regex_unknown_safety",
        ],
        "time_mode": mentions["time_mode"],
        "law_budget": DEFAULT_LAW_BUDGET,
        "court_budget": DEFAULT_COURT_BUDGET,
        "notes": "deterministic broad fallback planner",
    }


planner_tokenizer = None
planner_model = None


def load_planner_model():
    global planner_tokenizer, planner_model
    if planner_model is not None:
        return planner_tokenizer, planner_model
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    planner_tokenizer = AutoTokenizer.from_pretrained(PLANNER_MODEL_NAME, trust_remote_code=True)
    planner_model = AutoModelForCausalLM.from_pretrained(
        PLANNER_MODEL_NAME,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
    ).eval()
    return planner_tokenizer, planner_model


def compact_cards_for_prompt(cards: dict, top_law: int = 120, top_prefix: int = 80) -> dict:
    return {
        "law_codes": cards["law_codes"][:top_law],
        "law_unit_chains": cards["law_unit_chains"],
        "court_families": cards["court_families"],
        "bge_divisions": cards["bge_divisions"],
        "docket_prefixes": cards["docket_prefixes"][:top_prefix],
        "route_choices": cards["route_choices"],
        "time_choices": cards["time_choices"],
    }


def extract_first_json_object(text: str) -> dict:
    start = text.find("{")
    if start < 0:
        raise ValueError("No JSON object found")
    depth = 0
    for i in range(start, len(text)):
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                return loads_json(text[start : i + 1])
    raise ValueError("Unclosed JSON object")


def validate_plan(plan: dict, cards: dict) -> dict:
    allowed = {
        "law_codes": allowed_ids(cards["law_codes"]) | {"NO_CODE"},
        "law_unit_chains": allowed_ids(cards["law_unit_chains"]),
        "court_families": allowed_ids(cards["court_families"]) | {"unknown"},
        "bge_divisions": allowed_ids(cards["bge_divisions"]),
        "docket_prefixes": allowed_ids(cards["docket_prefixes"]),
        "routes": allowed_ids(cards["route_choices"]),
        "time_mode": allowed_ids(cards["time_choices"]),
    }
    clean = {}
    for key in ["law_codes", "law_unit_chains", "court_families", "bge_divisions", "docket_prefixes", "routes"]:
        values = plan.get(key) or []
        clean[key] = sorted({str(v) for v in values if str(v) in allowed[key]})
    time_mode = str(plan.get("time_mode") or "no_time_filter")
    clean["time_mode"] = time_mode if time_mode in allowed["time_mode"] else "no_time_filter"
    clean["law_budget"] = int(plan.get("law_budget") or DEFAULT_LAW_BUDGET)
    clean["court_budget"] = int(plan.get("court_budget") or DEFAULT_COURT_BUDGET)
    clean["notes"] = str(plan.get("notes") or "")
    return clean


def llm_plan(query: str, cards: dict) -> dict:
    tokenizer, model = load_planner_model()
    prompt_cards = compact_cards_for_prompt(cards)
    prompt = f'''
You are a Swiss legal retrieval planner. Choose only from the provided card IDs.
Do not output citations. Do not invent IDs.
Return strict JSON only with keys:
law_codes, law_unit_chains, court_families, bge_divisions, docket_prefixes, routes, time_mode, law_budget, court_budget, notes.

Important: dates in the facts are not court-year filters unless the query explicitly asks for a time period or recent/older law.

CHOICE_CARDS:
{dumps_json(prompt_cards)}

QUERY:
{query}
'''
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=4096,
        temperature=0.6,
        top_p=0.95,
        top_k=20,
        do_sample=True,
    )[0][inputs.input_ids.shape[1] :]
    raw = tokenizer.decode(output_ids, skip_special_tokens=True)
    try:
        plan = extract_first_json_object(raw)
    except Exception:
        repair_prompt = "Repair this planner output into strict JSON only. Output no prose.\n" + raw
        text = tokenizer.apply_chat_template([{"role": "user", "content": repair_prompt}], tokenize=False, add_generation_prompt=True, enable_thinking=False)
        inputs = tokenizer([text], return_tensors="pt").to(model.device)
        output_ids = model.generate(**inputs, max_new_tokens=2048, temperature=0.2, do_sample=False)[0][inputs.input_ids.shape[1] :]
        plan = extract_first_json_object(tokenizer.decode(output_ids, skip_special_tokens=True))
    return validate_plan(plan, cards)


def get_plan(query: str, cards: dict) -> dict:
    if USE_PLANNER_LLM:
        try:
            return llm_plan(query, cards)
        except Exception as exc:
            print("Planner failed, using heuristic fallback:", repr(exc))
    return validate_plan(heuristic_plan(query, cards), cards)

## 7. Candidate Expansion Helpers

In [10]:
def sql_quote_list(values: Iterable[str]) -> str:
    vals = [str(v).replace("'", "''") for v in values if v is not None]
    if not vals:
        return "('')"
    return "(" + ",".join(f"'{v}'" for v in vals) + ")"


def fetch_set(sql: str) -> set[str]:
    return set(con.execute(sql).fetchdf()["citation"].astype(str))


def create_temp_citation_table(name: str, citations: set[str] | list[str]) -> str:
    safe_name = re.sub(r"[^A-Za-z0-9_]", "_", name)
    df = pd.DataFrame({"citation": list(citations)})
    con.register("_tmp_candidate_df", df)
    con.execute(f"CREATE OR REPLACE TEMP TABLE {safe_name} AS SELECT citation FROM _tmp_candidate_df")
    con.unregister("_tmp_candidate_df")
    return safe_name


def expand_explicit_law_mentions(mentions: dict) -> set[str]:
    out = set()
    for m in mentions["laws"]:
        article = m.get("article")
        law_code = m.get("law_code")
        if not article:
            continue
        if law_code:
            out |= fetch_set(
                f'''
                SELECT citation
                FROM law_segments
                WHERE law_code = '{law_code.replace("'", "''")}'
                  AND article = '{article.replace("'", "''")}'
                '''
            )
        else:
            out |= fetch_set(
                f'''
                SELECT citation
                FROM law_segments
                WHERE article = '{article.replace("'", "''")}'
                '''
            )
    return out


def expand_explicit_court_mentions(mentions: dict) -> set[str]:
    out = set()
    for m in mentions["bge"]:
        base = m.get("base")
        if base:
            out |= fetch_set(
                f'''
                SELECT citation
                FROM court_segments
                WHERE court_base = '{base.replace("'", "''")}'
                '''
            )
    for m in mentions["dockets"]:
        docket = m.get("docket")
        if docket:
            out |= fetch_set(
                f'''
                SELECT citation
                FROM court_segments
                WHERE court_base = '{docket.replace("'", "''")}'
                   OR docket = '{docket.replace("'", "''")}'
                '''
            )
    return out


def selected_law_code_candidates(plan: dict, keep_no_code: bool = True) -> set[str]:
    codes = set(plan.get("law_codes") or [])
    codes.update(c for c in COMMON_SAFETY_LAW_CODES if c in allowed_ids(choice_cards["law_codes"]))
    clauses = []
    if codes:
        clauses.append(f"law_code IN {sql_quote_list(codes)}")
    if keep_no_code:
        clauses.append("law_code IS NULL")
    if not clauses:
        return set(LAW_SOURCE)
    return fetch_set("SELECT citation FROM law_segments WHERE " + " OR ".join(clauses))


def selected_unit_chain_candidates(base: set[str], plan: dict) -> set[str]:
    chains = set(plan.get("law_unit_chains") or [])
    if not chains:
        return set(base)
    chain_sql = sql_quote_list(chains)
    selected = fetch_set(
        f'''
        SELECT citation
        FROM law_segments
        WHERE coalesce(nullif(unit_chain, ''), 'article_only') IN {chain_sql}
        '''
    )
    return selected & set(base)


def selected_court_family_candidates(plan: dict) -> set[str]:
    families = set(plan.get("court_families") or ["court_bge", "court_case", "unknown"])
    clauses = []
    if "court_bge" in families:
        clauses.append("pattern = 'court_bge'")
    if "court_case" in families:
        clauses.append("pattern = 'court_case'")
    if "unknown" in families:
        clauses.append("pattern = 'unknown'")
    if not clauses:
        return set(COURT_SOURCE)
    return fetch_set("SELECT citation FROM court_segments WHERE " + " OR ".join(clauses))


def selected_court_domain_candidates(base: set[str], plan: dict) -> set[str]:
    divisions = set(plan.get("bge_divisions") or [])
    prefixes = set(plan.get("docket_prefixes") or [])
    clauses = []
    if divisions:
        clauses.append(f"(pattern = 'court_bge' AND bge_division IN {sql_quote_list(divisions)})")
    if prefixes:
        clauses.append(f"(pattern IN ('court_case', 'unknown') AND docket_prefix IN {sql_quote_list(prefixes)})")
    clauses.append("pattern = 'unknown'")
    if not clauses:
        return set(base)
    selected = fetch_set(
        f'''
        SELECT citation
        FROM court_segments
        WHERE {' OR '.join(clauses)}
        '''
    )
    return selected & set(base)


def same_decision_expansion(seed_court: set[str]) -> set[str]:
    if not seed_court:
        return set()
    seed_sql = sql_quote_list(seed_court)
    bases = con.execute(
        f'''
        SELECT DISTINCT court_base
        FROM court_segments
        WHERE citation IN {seed_sql}
        '''
    ).fetchdf()["court_base"].dropna().astype(str).tolist()
    if not bases:
        return set()
    return fetch_set(
        f'''
        SELECT citation
        FROM court_segments
        WHERE court_base IN {sql_quote_list(bases)}
        '''
    )


def statute_citing_court_candidates(selected_laws: set[str]) -> set[str]:
    if not selected_laws or "citation_edges" not in {r[0] for r in con.execute("SHOW TABLES").fetchall()}:
        return set()
    tmp = create_temp_citation_table("tmp_selected_laws_for_edges", selected_laws)
    return fetch_set(
        f'''
        SELECT DISTINCT e.source AS citation
        FROM citation_edges e
        JOIN court_segments c ON c.citation = e.source
        JOIN {tmp} l ON l.citation = e.target
        WHERE e.dataset = 'court_considerations'
        '''
    )


def unknown_regex_safety(plan: dict) -> set[str]:
    prefixes = set(plan.get("docket_prefixes") or [])
    if not prefixes:
        return fetch_set("SELECT citation FROM court_segments WHERE pattern = 'unknown'")
    return fetch_set(
        f'''
        SELECT citation
        FROM court_segments
        WHERE pattern = 'unknown'
          AND (docket_prefix IN {sql_quote_list(prefixes)} OR raw IS NOT NULL)
        '''
    )

## 8. Recall Audit And Budgeting

In [11]:
audit_rows = []
dropped_rows = []


def split_gold_by_funnel(gold_set: set[str]) -> tuple[set[str], set[str]]:
    return gold_set & LAW_SOURCE, gold_set & COURT_SOURCE


def recall_of(candidates: set[str], gold: set[str]) -> float | None:
    if not gold:
        return None
    return len(candidates & gold) / len(gold)


def audit_stage(
    query_id: str,
    funnel: str,
    stage: str,
    before: set[str],
    after: set[str],
    gold: set[str],
    min_recall: float | None,
    drop_reason: str,
) -> set[str]:
    """Diagnostic-only auditor. Never reverts a stage based on gold.

    Earlier versions reverted to `before` when `recall_after < min_recall` on
    val. That is a gold-aware decision that does not generalize to test, so
    it has been removed. We still record `min_recall` and `status` so plots
    can show which stages would have been guarded.
    """
    before_gold = before & gold
    after_gold = after & gold
    dropped = sorted(before_gold - after_gold)
    rec_before = recall_of(before, gold)
    rec_after = recall_of(after, gold)
    status = "ok"
    if gold and min_recall is not None and (rec_after or 0.0) < min_recall:
        status = "below_min_recall"  # diagnostic; do NOT revert

    audit_rows.append(
        {
            "query_id": query_id,
            "funnel": funnel,
            "stage": stage,
            "candidate_count_before": len(before),
            "candidate_count_after": len(after),
            "candidate_count_final": len(after),
            "gold_count": len(gold),
            "gold_kept_before": len(before_gold),
            "gold_kept_after": len(after_gold),
            "gold_dropped": len(dropped),
            "recall_before": rec_before,
            "recall_after": rec_after,
            "min_recall": min_recall,
            "status": status,
            "drop_reason": drop_reason,
        }
    )

    for citation in dropped:
        dropped_rows.append(
            {
                "query_id": query_id,
                "funnel": funnel,
                "first_failed_stage": stage,
                "citation": citation,
                "drop_reason": drop_reason,
                "stage_status": status,
            }
        )

    return after


def score_law_candidates(candidates: set[str], query: str, plan: dict, explicit: set[str]) -> pd.DataFrame:
    if not candidates:
        return pd.DataFrame(columns=["citation", "score"])
    tmp = create_temp_citation_table("tmp_law_score_candidates", candidates)
    df = con.execute(
        f"""
        SELECT s.citation, s.law_code, s.article, s.unit_chain, coalesce(l.title, '') AS title
        FROM {tmp} tc
        JOIN law_segments s USING (citation)
        LEFT JOIN law_citations l USING (citation)
        """
    ).fetchdf()
    selected_codes = set(plan.get("law_codes") or [])
    q = query[:2000]
    scores = []
    for row in df.itertuples(index=False):
        score = 0.0
        if row.citation in explicit:
            score += 1000
        if row.law_code in selected_codes:
            score += 80
        if row.law_code in COMMON_SAFETY_LAW_CODES:
            score += 20
        if row.title:
            score += 0.25 * fuzz.partial_ratio(q, str(row.title))
        if row.article and str(row.article) in q:
            score += 15
        scores.append(score)
    df["score"] = scores
    return df[["citation", "score"]].sort_values(["score", "citation"], ascending=[False, True])


def score_court_candidates(candidates: set[str], query: str, plan: dict, explicit: set[str], citing_cases: set[str]) -> pd.DataFrame:
    if not candidates:
        return pd.DataFrame(columns=["citation", "score"])
    tmp = create_temp_citation_table("tmp_court_score_candidates", candidates)
    df = con.execute(
        f"""
        SELECT c.citation, c.pattern, c.subfamily, c.docket_prefix, c.bge_division, c.decision_year, c.court_base, c.consideration
        FROM {tmp} tc
        JOIN court_segments c USING (citation)
        """
    ).fetchdf()
    prefixes = set(plan.get("docket_prefixes") or [])
    divisions = set(plan.get("bge_divisions") or [])
    time_mode = plan.get("time_mode") or "no_time_filter"
    q = query[:2000]
    scores = []
    for row in df.itertuples(index=False):
        score = 0.0
        if row.citation in explicit:
            score += 1000
        if row.citation in citing_cases:
            score += 120
        if row.pattern == "court_bge":
            score += 40
        if row.pattern == "court_case":
            score += 30
        if row.pattern == "unknown":
            score += 5
        if row.docket_prefix in prefixes:
            score += 80
        if row.bge_division in divisions:
            score += 80
        if row.court_base and str(row.court_base) in q:
            score += 500
        if time_mode == "soft_recency" and row.decision_year and str(row.decision_year).isdigit():
            score += max(0, int(str(row.decision_year)[-4:]) - 1990) / 10
        scores.append(score)
    df["score"] = scores
    return df[["citation", "score"]].sort_values(["score", "citation"], ascending=[False, True])


def cap_candidates_with_guard(
    query_id: str,
    funnel: str,
    stage: str,
    before: set[str],
    scored: pd.DataFrame,
    budget: int,
    gold: set[str],
    explicit_keep: set[str],
    min_recall: float,
) -> set[str]:
    """Hard-cap to `budget`, pin `explicit_keep`. Never grows past budget on
    gold-aware recall checks. Audit only records whether recall fell below
    `min_recall` so it can be inspected after the run."""
    if len(before) <= budget:
        return before
    budget = max(0, int(budget))
    top = set(scored.head(budget)["citation"].astype(str)) | explicit_keep
    return audit_stage(query_id, funnel, stage, before, top, gold, min_recall, "budget_cut")


## 9. Funnel Runner

## 9b. Soft-Rank Funnel (Recall-Aware Top-K)

The hard-filter funnel above either passes everything or risks dropping gold, so the recall guard often reverts entire stages and the pool stays at corpus scale. The soft variant below replaces hard filters with a *score* and uses a single recall-aware top-K cap. Pinned items (explicit query mentions, planner-aligned domain matches) get large score boosts and are guaranteed to be kept. If the top-K pool fails the recall gate on val, K is grown along `SOFT_PROBE_STEPS` until the gate passes or `MAX_TOTAL_CANDIDATES` is reached.

In [12]:
# --- Soft-rank scoring (whole corpus, then top-K) ---------------------------
#
# Score every source row, pin must-keep items, take top-K. K is fixed by the
# planner budget; we never grow K against gold (that would be a val-only leak).

LAW_SCORE_SQL_TEMPLATE = """
SELECT
    s.citation,
    (
        CASE WHEN s.citation IN {explicit_sql} THEN 1000.0 ELSE 0.0 END
      + CASE WHEN s.law_code IN {selected_codes_sql} THEN 80.0 ELSE 0.0 END
      + CASE WHEN s.law_code IN {safety_codes_sql} THEN 20.0 ELSE 0.0 END
      + CASE WHEN s.law_code IS NULL THEN 5.0 ELSE 0.0 END
      + CASE WHEN s.article IN {explicit_articles_sql} THEN 25.0 ELSE 0.0 END
      + CASE WHEN s.citation IN {train_pin_sql} THEN 60.0 ELSE 0.0 END
    ) AS score
FROM law_segments s
"""

COURT_SCORE_SQL_TEMPLATE = """
WITH citing AS (
    SELECT DISTINCT e.source AS citation
    FROM citation_edges e
    JOIN ({selected_laws_sql_inner}) l ON l.citation = e.target
    WHERE e.dataset = 'court_considerations'
)
SELECT
    c.citation,
    (
        CASE WHEN c.citation IN {explicit_sql} THEN 1000.0 ELSE 0.0 END
      + CASE WHEN c.court_base IN {explicit_bases_sql} THEN 800.0 ELSE 0.0 END
      + CASE WHEN c.docket_prefix IN {selected_prefixes_sql} THEN 80.0 ELSE 0.0 END
      + CASE WHEN c.bge_division IN {selected_divisions_sql} AND c.pattern = 'court_bge' THEN 80.0 ELSE 0.0 END
      + CASE WHEN c.pattern = 'court_bge' THEN 25.0 ELSE 0.0 END
      + CASE WHEN c.pattern = 'court_case' THEN 15.0 ELSE 0.0 END
      + CASE WHEN c.pattern = 'unknown'   THEN 2.0  ELSE 0.0 END
      + CASE WHEN c.citation IN (SELECT citation FROM citing) THEN 120.0 ELSE 0.0 END
      + CASE WHEN c.citation IN {train_pin_sql} THEN 60.0 ELSE 0.0 END
    ) AS score
FROM court_segments c
"""

def _sql_in_or_empty(values: Iterable[str]) -> str:
    """Quoted IN list that always matches nothing if values is empty."""
    vals = [str(v).replace("'", "''") for v in values if v is not None and str(v) != ""]
    if not vals:
        return "('__no_match_sentinel__')"
    return "(" + ",".join(f"'{v}'" for v in vals) + ")"


# Train-only prior. Loaded once. This is fair on every split because train gold
# IS training data. It is NOT the gold-bank leakage of using val/test gold.
def _load_train_gold_pin() -> tuple[set[str], set[str]]:
    path = INSIGHTS_DIR / "train_gold_citations.json"
    if not path.exists():
        print("train_gold_citations.json not found; train pin disabled")
        return set(), set()
    bank = set(loads_json(path.read_bytes()))
    return bank & LAW_SOURCE, bank & COURT_SOURCE


TRAIN_LAW_PIN, TRAIN_COURT_PIN = _load_train_gold_pin()
print(f"Train-prior pin (legit): law={len(TRAIN_LAW_PIN):,}, court={len(TRAIN_COURT_PIN):,}")


def score_all_law_rows(plan: dict, mentions: dict, explicit: set[str]) -> pd.DataFrame:
    selected_codes = set(plan.get("law_codes") or [])
    safety_codes = COMMON_SAFETY_LAW_CODES & allowed_ids(choice_cards["law_codes"])
    explicit_articles = {m.get("article") for m in mentions.get("laws", []) if m.get("article")}

    sql = LAW_SCORE_SQL_TEMPLATE.format(
        explicit_sql=_sql_in_or_empty(explicit),
        selected_codes_sql=_sql_in_or_empty(selected_codes),
        safety_codes_sql=_sql_in_or_empty(safety_codes),
        explicit_articles_sql=_sql_in_or_empty(explicit_articles),
        train_pin_sql=_sql_in_or_empty(TRAIN_LAW_PIN),
    )
    return con.execute(sql).fetchdf()


def score_all_court_rows(plan: dict, mentions: dict, explicit: set[str], selected_laws: set[str]) -> pd.DataFrame:
    selected_prefixes = set(plan.get("docket_prefixes") or [])
    selected_divisions = set(plan.get("bge_divisions") or [])
    explicit_bases = set()
    for m in mentions.get("bge", []):
        if m.get("base"):
            explicit_bases.add(m["base"])
    for m in mentions.get("dockets", []):
        if m.get("docket"):
            explicit_bases.add(m["docket"])

    if selected_laws:
        tmp = create_temp_citation_table("tmp_soft_selected_laws", selected_laws)
        selected_laws_sql_inner = f"SELECT citation FROM {tmp}"
    else:
        selected_laws_sql_inner = "SELECT '__no_match_sentinel__' AS citation WHERE 1=0"

    sql = COURT_SCORE_SQL_TEMPLATE.format(
        explicit_sql=_sql_in_or_empty(explicit),
        explicit_bases_sql=_sql_in_or_empty(explicit_bases),
        selected_prefixes_sql=_sql_in_or_empty(selected_prefixes),
        selected_divisions_sql=_sql_in_or_empty(selected_divisions),
        selected_laws_sql_inner=selected_laws_sql_inner,
        train_pin_sql=_sql_in_or_empty(TRAIN_COURT_PIN),
    )
    return con.execute(sql).fetchdf()


def take_top_k_with_recall_probe(
    scored: pd.DataFrame,
    pinned: set[str],
    base_k: int,
    gold: set[str],
    min_recall: float,
    max_k: int,
) -> tuple[set[str], int, float | None, str]:
    """Take top-K by score, union with pinned. K is fixed; we never grow K
    based on gold (that would be a val-only leak).

    Returns (chosen_set, final_k, recall_after, status). `recall_after` is
    None when gold is empty; otherwise it is reported for diagnostics only.
    """
    if scored.empty:
        return set(pinned), len(pinned), recall_of(set(pinned), gold) if gold else None, "no_candidates"

    scored_sorted = scored.sort_values(["score", "citation"], ascending=[False, True])
    citations_sorted = scored_sorted["citation"].astype(str).tolist()

    k = int(min(base_k, len(citations_sorted), max_k))
    chosen = set(citations_sorted[:k]) | set(pinned)
    rec = recall_of(chosen, gold) if gold else None
    return chosen, len(chosen), rec, "ok"


def run_law_funnel_soft(query_id: str, query: str, gold_law: set[str], plan: dict, mentions: dict) -> tuple[set[str], dict]:
    explicit = expand_explicit_law_mentions(mentions)
    pinned = set(explicit) | set(TRAIN_LAW_PIN)

    scored = score_all_law_rows(plan, mentions, explicit)

    chosen, k, rec, status = take_top_k_with_recall_probe(
        scored=scored,
        pinned=pinned,
        base_k=int(plan.get("law_budget") or SOFT_LAW_TARGET),
        gold=gold_law,
        min_recall=FINAL_MIN_QUERY_RECALL,
        max_k=SOFT_LAW_TARGET,
    )

    audit_rows.append({
        "query_id": query_id, "funnel": "law", "stage": "L_soft_topk",
        "candidate_count_before": len(LAW_SOURCE),
        "candidate_count_after": k, "candidate_count_final": k,
        "gold_count": len(gold_law),
        "gold_kept_before": len(LAW_SOURCE & gold_law),
        "gold_kept_after": len(chosen & gold_law),
        "gold_dropped": len((LAW_SOURCE & gold_law) - (chosen & gold_law)),
        "recall_before": 1.0 if gold_law else None,
        "recall_after": rec,
        "min_recall": FINAL_MIN_QUERY_RECALL,
        "status": status,
        "drop_reason": "soft_topk",
    })
    diag = {
        "explicit_law_candidates": sorted(explicit),
        "stages": {"L_soft_topk": k},
        "soft_status": status,
        "candidate_count": k,
        "recall": rec,
    }
    return chosen, diag


def run_court_funnel_soft(query_id: str, query: str, gold_court: set[str], plan: dict, mentions: dict, selected_laws: set[str]) -> tuple[set[str], dict]:
    explicit = expand_explicit_court_mentions(mentions)
    pinned = set(explicit) | set(TRAIN_COURT_PIN)

    # Pin one same-decision-neighbors hop from explicit BGE/docket bases.
    if explicit:
        pinned |= same_decision_expansion(explicit)

    scored = score_all_court_rows(plan, mentions, explicit, selected_laws)

    chosen, k, rec, status = take_top_k_with_recall_probe(
        scored=scored,
        pinned=pinned,
        base_k=int(plan.get("court_budget") or SOFT_COURT_TARGET),
        gold=gold_court,
        min_recall=FINAL_MIN_QUERY_RECALL,
        max_k=SOFT_COURT_TARGET,
    )

    audit_rows.append({
        "query_id": query_id, "funnel": "court", "stage": "C_soft_topk",
        "candidate_count_before": len(COURT_SOURCE),
        "candidate_count_after": k, "candidate_count_final": k,
        "gold_count": len(gold_court),
        "gold_kept_before": len(COURT_SOURCE & gold_court),
        "gold_kept_after": len(chosen & gold_court),
        "gold_dropped": len((COURT_SOURCE & gold_court) - (chosen & gold_court)),
        "recall_before": 1.0 if gold_court else None,
        "recall_after": rec,
        "min_recall": FINAL_MIN_QUERY_RECALL,
        "status": status,
        "drop_reason": "soft_topk",
    })
    diag = {
        "explicit_court_candidates": sorted(explicit),
        "pinned_count": len(pinned),
        "stages": {"C_soft_topk": k},
        "soft_status": status,
        "candidate_count": k,
        "recall": rec,
    }
    return chosen, diag


def merge_candidate_sets_soft(query_id: str, law_candidates: set[str], court_candidates: set[str], gold_set: set[str]) -> set[str]:
    """Merge with a hard cap at MAX_TOTAL_CANDIDATES. Splits the cap evenly
    by current side sizes (laws first, then courts in a stable sort). Never
    refuses to trim based on gold.
    """
    merged = set(law_candidates) | set(court_candidates)
    if len(merged) <= MAX_TOTAL_CANDIDATES:
        return merged

    overflow = len(merged) - MAX_TOTAL_CANDIDATES
    trimmed_court = set(sorted(court_candidates)[: max(0, len(court_candidates) - overflow)])
    return set(law_candidates) | trimmed_court


print("Soft funnel helpers loaded (no-leak version).")
print("SOFT_LAW_TARGET    =", SOFT_LAW_TARGET)
print("SOFT_COURT_TARGET  =", SOFT_COURT_TARGET)
print("MAX_TOTAL_CANDIDATES =", MAX_TOTAL_CANDIDATES)


Train-prior pin (legit): law=1,876, court=0
Soft funnel helpers loaded (no-leak version).
SOFT_LAW_TARGET    = 2000
SOFT_COURT_TARGET  = 2000
MAX_TOTAL_CANDIDATES = 4000


In [13]:
def run_law_funnel(query_id: str, query: str, gold_law: set[str], plan: dict, mentions: dict) -> tuple[set[str], dict]:
    explicit = expand_explicit_law_mentions(mentions)
    stages = {}

    s0 = set(LAW_SOURCE)
    stages["L0_all_laws"] = s0
    s0 = audit_stage(query_id, "law", "L0_all_laws", s0, s0, gold_law, 1.0 if gold_law else None, "baseline")

    s1_raw = selected_law_code_candidates(plan, keep_no_code=True)
    s1 = s1_raw | explicit
    s1 = audit_stage(query_id, "law", "L1_domain_or_law_family", s0, s1, gold_law, EARLY_STAGE_MIN_RECALL, "planner_missed_choice")
    stages["L1_domain_or_law_family"] = s1

    s2_raw = selected_law_code_candidates(plan, keep_no_code=True)
    s2 = s2_raw | explicit
    s2 = audit_stage(query_id, "law", "L2_law_code", s1, s2, gold_law, EARLY_STAGE_MIN_RECALL, "planner_missed_choice")
    stages["L2_law_code"] = s2

    article_group = explicit if explicit else s2
    s3 = audit_stage(query_id, "law", "L3_article_group", s2, article_group, gold_law, MID_STAGE_MIN_RECALL, "filter_too_strict")
    stages["L3_article_group"] = s3

    s4_raw = selected_unit_chain_candidates(s3, plan)
    s4 = s4_raw | explicit
    s4 = audit_stage(query_id, "law", "L4_unit_depth", s3, s4, gold_law, MID_STAGE_MIN_RECALL, "filter_too_strict")
    stages["L4_unit_depth"] = s4

    scored = score_law_candidates(s4, query, plan, explicit)
    s5 = cap_candidates_with_guard(
        query_id,
        "law",
        "L5_budget",
        s4,
        scored,
        int(plan.get("law_budget") or DEFAULT_LAW_BUDGET),
        gold_law,
        explicit,
        FINAL_MIN_QUERY_RECALL,
    )
    stages["L5_budget"] = s5
    return s5, {"explicit_law_candidates": sorted(explicit), "stages": {k: len(v) for k, v in stages.items()}}


def run_court_funnel(
    query_id: str,
    query: str,
    gold_court: set[str],
    plan: dict,
    mentions: dict,
    selected_laws: set[str],
) -> tuple[set[str], dict]:
    explicit = expand_explicit_court_mentions(mentions)
    stages = {}

    s0 = set(COURT_SOURCE)
    stages["C0_all_court"] = s0
    s0 = audit_stage(query_id, "court", "C0_all_court", s0, s0, gold_court, 1.0 if gold_court else None, "baseline")

    s1_raw = selected_court_family_candidates(plan)
    s1 = s1_raw | explicit
    s1 = audit_stage(query_id, "court", "C1_court_family", s0, s1, gold_court, EARLY_STAGE_MIN_RECALL, "planner_missed_choice")
    stages["C1_court_family"] = s1

    s2_raw = selected_court_domain_candidates(s1, plan)
    s2 = s2_raw | explicit
    s2 = audit_stage(query_id, "court", "C2_domain_prefix", s1, s2, gold_court, EARLY_STAGE_MIN_RECALL, "planner_missed_choice")
    stages["C2_domain_prefix"] = s2

    decision_seed = explicit
    if decision_seed:
        decision_expanded = same_decision_expansion(decision_seed)
        s3_raw = (s2 & decision_expanded) | decision_expanded | explicit
    else:
        s3_raw = s2
    s3 = audit_stage(query_id, "court", "C3_decision_level", s2, s3_raw, gold_court, MID_STAGE_MIN_RECALL, "route_missing")
    stages["C3_decision_level"] = s3

    citing_cases = statute_citing_court_candidates(selected_laws)
    s4_raw = (s3 | citing_cases | explicit)
    s4 = audit_stage(query_id, "court", "C4_statute_citing_cases", s3, s4_raw, gold_court, MID_STAGE_MIN_RECALL, "route_missing")
    stages["C4_statute_citing_cases"] = s4

    unknown_safety = unknown_regex_safety(plan)
    s5_raw = s4 | unknown_safety | explicit
    s5 = audit_stage(query_id, "court", "C5_regex_and_unknown_safety", s4, s5_raw, gold_court, MID_STAGE_MIN_RECALL, "regex_gap")
    stages["C5_regex_and_unknown_safety"] = s5

    scored = score_court_candidates(s5, query, plan, explicit, citing_cases)
    s6 = cap_candidates_with_guard(
        query_id,
        "court",
        "C6_budget",
        s5,
        scored,
        int(plan.get("court_budget") or DEFAULT_COURT_BUDGET),
        gold_court,
        explicit,
        FINAL_MIN_QUERY_RECALL,
    )
    stages["C6_budget"] = s6
    return s6, {
        "explicit_court_candidates": sorted(explicit),
        "citing_case_count": len(citing_cases),
        "unknown_safety_count": len(unknown_safety),
        "stages": {k: len(v) for k, v in stages.items()},
    }


def merge_candidate_sets(query_id: str, law_candidates: set[str], court_candidates: set[str], gold_set: set[str]) -> set[str]:
    merged = set(law_candidates) | set(court_candidates)
    if len(merged) <= MAX_TOTAL_CANDIDATES:
        return merged

    # Hard funnel already budgeted each side. If union exceeds the global cap,
    # trim from the court side stably. We never look at gold to decide.
    overflow = len(merged) - MAX_TOTAL_CANDIDATES
    trimmed_court = set(sorted(court_candidates)[: max(0, len(court_candidates) - overflow)])
    return set(law_candidates) | trimmed_court


## 10. Run Candidate Funnel

## 9c. Gold-Bank Candidate Funnel (LEAKAGE — diagnostic only, OFF by default)

This path uses `data_insights/{split}_gold_citations.json` directly as the
candidate set. With `["val"]` it trivially yields recall 1.0 on val because
the candidate list IS the val gold. With `["train", "val"]` it leaks val
labels into the val candidate pool. It is therefore **not a retrieval method**
and is kept only as a diagnostic for the per-side candidate plumbing.

`USE_GOLD_BANK_CANDIDATES` is `False` by default. Do not turn it on for any
val or test recall claim you intend to take seriously.


In [14]:
# --- Gold-bank candidate path ----------------------------------------------
#
# This is intentionally separate from the structural/soft funnel above. It is
# useful for recall auditing and for creating a very small closed-vocabulary
# candidate bank from the helper files in data_insights.

GOLD_BANK_CACHE = {}


def active_gold_bank_splits() -> list[str]:
    if RUN_SPLIT == "val":
        return list(GOLD_BANK_SPLITS_FOR_VAL)
    return list(GOLD_BANK_SPLITS_FOR_TEST)


def load_gold_bank(split_names: list[str] | tuple[str, ...]) -> dict:
    key = tuple(split_names)
    if key in GOLD_BANK_CACHE:
        return GOLD_BANK_CACHE[key]

    bank = set()
    missing = []
    for split in key:
        path = INSIGHTS_DIR / f"{split}_gold_citations.json"
        if not path.exists():
            missing.append(str(path))
            continue
        bank.update(loads_json(path.read_bytes()))

    law_bank = bank & LAW_SOURCE
    court_bank = bank & COURT_SOURCE
    if len(law_bank) > GOLD_BANK_LAW_CAP:
        raise ValueError(f"Gold-bank law candidates exceed cap: {len(law_bank)} > {GOLD_BANK_LAW_CAP}")
    if len(court_bank) > GOLD_BANK_COURT_CAP:
        raise ValueError(f"Gold-bank court candidates exceed cap: {len(court_bank)} > {GOLD_BANK_COURT_CAP}")

    out = {
        "splits": list(key),
        "all": bank,
        "law": law_bank,
        "court": court_bank,
        "missing_files": missing,
    }
    GOLD_BANK_CACHE[key] = out
    print(
        "Gold bank loaded:",
        ",".join(key),
        "law=", len(law_bank),
        "court=", len(court_bank),
        "missing_files=", len(missing),
    )
    return out


def run_gold_bank_candidate_funnel(
    query_id: str,
    gold_law: set[str],
    gold_court: set[str],
) -> tuple[set[str], set[str], dict]:
    bank = load_gold_bank(active_gold_bank_splits())
    law_candidates = set(bank["law"])
    court_candidates = set(bank["court"])

    return law_candidates, court_candidates, {
        "gold_bank_splits": bank["splits"],
        "missing_files": bank["missing_files"],
        "law_diag": {
            "soft_status": "gold_bank",
            "candidate_count": len(law_candidates),
            "recall": recall_of(law_candidates, gold_law),
        },
        "court_diag": {
            "soft_status": "gold_bank",
            "candidate_count": len(court_candidates),
            "recall": recall_of(court_candidates, gold_court),
        },
    }


if USE_GOLD_BANK_CANDIDATES:
    _preview_bank = load_gold_bank(active_gold_bank_splits())
    print("USE_GOLD_BANK_CANDIDATES = True")
    print("law candidates  :", len(_preview_bank["law"]))
    print("court candidates:", len(_preview_bank["court"]))

In [15]:
candidate_output_path = ART_DIR / f"{RUN_SPLIT}_candidate_sets.jsonl"
plan_output_path = ART_DIR / f"{RUN_SPLIT}_planner_outputs.jsonl"

candidate_rows = []
plan_rows = []
audit_rows.clear()
dropped_rows.clear()

with candidate_output_path.open("w", encoding="utf-8") as cand_out, plan_output_path.open("w", encoding="utf-8") as plan_out:
    for row in queries.itertuples(index=False):
        query_id = str(getattr(row, "query_id"))
        query = str(getattr(row, "query"))
        gold_set = set(getattr(row, "gold_set", set()) or set())
        gold_law, gold_court = split_gold_by_funnel(gold_set)

        mentions = extract_query_mentions(query)
        if USE_GOLD_BANK_CANDIDATES:
            plan = {
                "notes": "gold_bank_candidate_funnel",
                "gold_bank_splits": active_gold_bank_splits(),
                "law_budget": GOLD_BANK_LAW_CAP,
                "court_budget": GOLD_BANK_COURT_CAP,
            }
        else:
            plan = get_plan(query, choice_cards)
        plan_rows.append({"query_id": query_id, "plan": plan, "mentions": mentions})
        plan_out.write(dumps_json({"query_id": query_id, "plan": plan, "mentions": mentions}) + "\n")

        print("\n===", query_id, "===")
        print("gold law/court:", len(gold_law), len(gold_court))
        print("plan:", plan)

        if USE_GOLD_BANK_CANDIDATES:
            law_candidates, court_candidates, gold_bank_diag = run_gold_bank_candidate_funnel(query_id, gold_law, gold_court)
            law_diag = gold_bank_diag["law_diag"]
            court_diag = gold_bank_diag["court_diag"]
            merged = set(law_candidates) | set(court_candidates)
        elif SOFT_FUNNEL_ENABLED:
            law_candidates, law_diag = run_law_funnel_soft(query_id, query, gold_law, plan, mentions)
            court_candidates, court_diag = run_court_funnel_soft(query_id, query, gold_court, plan, mentions, law_candidates)
            merged = merge_candidate_sets_soft(query_id, law_candidates, court_candidates, gold_set)
        else:
            law_candidates, law_diag = run_law_funnel(query_id, query, gold_law, plan, mentions)
            court_candidates, court_diag = run_court_funnel(query_id, query, gold_court, plan, mentions, law_candidates)
            merged = merge_candidate_sets(query_id, law_candidates, court_candidates, gold_set)

        record = {
            "query_id": query_id,
            "candidate_count": len(merged),
            "law_candidate_count": len(law_candidates),
            "court_candidate_count": len(court_candidates),
            "recall": recall_of(merged, gold_set),
            "law_recall": recall_of(law_candidates, gold_law),
            "court_recall": recall_of(court_candidates, gold_court),
            "candidates": sorted(merged),
            "law_candidates": sorted(law_candidates),
            "court_candidates": sorted(court_candidates),
            "law_diag": law_diag,
            "court_diag": court_diag,
        }
        candidate_rows.append(record)
        cand_out.write(dumps_json(record) + "\n")
        print(
            "candidate_count:", len(merged),
            "law_n:", len(law_candidates),
            "court_n:", len(court_candidates),
            "recall:", record["recall"],
            "law:", record["law_recall"],
            "court:", record["court_recall"],
            "law_status:", law_diag.get("soft_status"),
            "court_status:", court_diag.get("soft_status"),
        )

print("Saved candidates:", candidate_output_path)
print("Saved plans     :", plan_output_path)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 17 files:   0%|          | 0/17 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/707 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]


=== val_001 ===
gold law/court: 19 12
plan: {'law_codes': ['StPO'], 'law_unit_chains': ['article_only'], 'court_families': [], 'bge_divisions': [], 'docket_prefixes': ['2C'], 'routes': ['bge_base_route', 'law_code_route', 'statute_citing_cases'], 'time_mode': 'no_time_filter', 'law_budget': 2000, 'court_budget': 2000, 'notes': "Art. 221 Abs. 1 lit. b StPO permits pre-trial detention extensions to prevent collusion or evidence tampering. Proportionality requires balancing the risk of release against the detainee's rights. Key factors: (1) prosecutor's concrete risk allegations (witness influence, reoffending), (2) defense arguments about completed witness interviews and technical pending steps, (3) victim's withdrawn complaint. The court must assess whether the asserted risks outweigh the public interest in liberty and the adequacy of less restrictive measures."}
candidate_count: 4000 law_n: 3119 court_n: 2000 recall: 0.38095238095238093 law: 0.8421052631578947 court: 0.0 law_status: o

## 11. Recall Diagnostics

In [16]:
audit_df = pd.DataFrame(audit_rows)
dropped_df = pd.DataFrame(dropped_rows)
summary_df = pd.DataFrame(
    [
        {
            "query_id": r["query_id"],
            "candidate_count": r["candidate_count"],
            "law_candidate_count": r["law_candidate_count"],
            "court_candidate_count": r["court_candidate_count"],
            "recall": r["recall"],
            "law_recall": r["law_recall"],
            "court_recall": r["court_recall"],
        }
        for r in candidate_rows
    ]
)

audit_path = ART_DIR / "stage_recall_audit.csv"
dropped_path = ART_DIR / "dropped_gold_diagnostics.csv"
summary_path = ART_DIR / f"{RUN_SPLIT}_candidate_summary.csv"

audit_df.to_csv(audit_path, index=False)
dropped_df.to_csv(dropped_path, index=False)
summary_df.to_csv(summary_path, index=False)

print("Saved:", audit_path)
print("Saved:", dropped_path)
print("Saved:", summary_path)

display(summary_df)
if not audit_df.empty:
    display(
        audit_df.groupby(["funnel", "stage", "status"], dropna=False)
        .agg(
            queries=("query_id", "nunique"),
            mean_after_count=("candidate_count_final", "mean"),
            min_recall_after=("recall_after", "min"),
            mean_recall_after=("recall_after", "mean"),
            total_dropped=("gold_dropped", "sum"),
        )
        .reset_index()
    )

if RUN_SPLIT == "val" and not summary_df.empty:
    print("Mean recall:", summary_df["recall"].mean())
    print("Min recall :", summary_df["recall"].min())
    print("Mean law recall:", summary_df["law_recall"].dropna().mean())
    print("Mean court recall:", summary_df["court_recall"].dropna().mean())
    bad = summary_df[summary_df["recall"].fillna(1.0) < FINAL_MIN_QUERY_RECALL]
    if len(bad):
        print("Queries below final gate:")
        display(bad)
    else:
        print("All validation queries pass the final per-query recall gate.")

Saved: /content/drive/MyDrive/swiss_law/artifacts/stage_recall_audit.csv
Saved: /content/drive/MyDrive/swiss_law/artifacts/dropped_gold_diagnostics.csv
Saved: /content/drive/MyDrive/swiss_law/artifacts/val_candidate_summary.csv


,query_id,candidate_count,law_candidate_count,court_candidate_count,recall,law_recall,court_recall
0,val_001,4000,3119,2000,0.380952,0.842105,0.0
1,val_002,1879,1876,3,0.250000,0.450000,0.0
2,val_003,4000,3710,2000,0.382979,0.750000,0.0
3,val_004,1879,1876,3,0.300000,0.333333,NaN
4,val_005,1881,1876,5,0.181818,0.333333,NaN
5,val_006,1886,1876,10,0.333333,0.545455,NaN
6,val_007,1884,1882,2,0.578947,0.733333,NaN
7,val_008,1877,1876,1,0.275862,0.400000,0.0
8,val_009,1878,1876,2,0.285714,0.363636,0.0
9,val_010,1879,1876,3,0.080000,0.142857,0.0


,funnel,stage,status,queries,mean_after_count,min_recall_after,mean_recall_after,total_dropped
0,court,C_soft_topk,ok,10,402.9,0.000000,0.000000,32
1,law,L_soft_topk,ok,10,2184.3,0.142857,0.489405,70


Mean recall: 0.3049606342609007
Min recall : 0.08
Mean law recall: 0.4894053315105946
Mean court recall: 0.0
Queries below final gate:


,query_id,candidate_count,law_candidate_count,court_candidate_count,recall,law_recall,court_recall
0,val_001,4000,3119,2000,0.380952,0.842105,0.0
1,val_002,1879,1876,3,0.250000,0.450000,0.0
2,val_003,4000,3710,2000,0.382979,0.750000,0.0
3,val_004,1879,1876,3,0.300000,0.333333,NaN
4,val_005,1881,1876,5,0.181818,0.333333,NaN
5,val_006,1886,1876,10,0.333333,0.545455,NaN
6,val_007,1884,1882,2,0.578947,0.733333,NaN
7,val_008,1877,1876,1,0.275862,0.400000,0.0
8,val_009,1878,1876,2,0.285714,0.363636,0.0
9,val_010,1879,1876,3,0.080000,0.142857,0.0


## 12. Inspect First Dropped Gold Citations

Use this before tightening any filter. A filter is allowed only if the dropped citation has a clear recovery route or the drop is accepted as rare.

In [17]:
if dropped_df.empty:
    print("No gold citations were dropped by audited stages, or no gold labels are available.")
else:
    display(dropped_df.head(100))
    display(dropped_df.groupby(["funnel", "first_failed_stage", "drop_reason", "stage_status"]).size().reset_index(name="n"))

No gold citations were dropped by audited stages, or no gold labels are available.


## 13. Optional Reranker Skeleton

Run this only after candidate recall is acceptable. Reranking is not allowed to add invented citations; it can only score existing candidate IDs.

In [18]:
reranker_tokenizer = None
reranker_model = None


def load_reranker_model():
    global reranker_tokenizer, reranker_model
    if reranker_model is not None:
        return reranker_tokenizer, reranker_model
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    reranker_tokenizer = AutoTokenizer.from_pretrained(RERANKER_MODEL_NAME, padding_side="left", trust_remote_code=True)
    reranker_model = AutoModelForCausalLM.from_pretrained(
        RERANKER_MODEL_NAME,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
    ).eval()
    return reranker_tokenizer, reranker_model


def candidate_evidence_card(citation: str) -> str:
    if citation in LAW_SOURCE:
        row = con.execute(
            f'''
            SELECT s.citation, s.law_code, s.article, s.unit_chain, coalesce(l.title, '') AS title, left(coalesce(l.text, ''), 1200) AS text
            FROM law_segments s
            LEFT JOIN law_citations l USING (citation)
            WHERE s.citation = '{citation.replace("'", "''")}'
            LIMIT 1
            '''
        ).fetchone()
        if row:
            return f"Citation: {row[0]}\nType: law\nLaw code: {row[1]}\nArticle: {row[2]}\nUnits: {row[3]}\nTitle: {row[4]}\nText: {row[5]}"
    row = con.execute(
        f'''
        SELECT c.citation, c.pattern, c.subfamily, c.court_base, c.docket_prefix, c.bge_division, c.decision_year,
               left(coalesce(t.text, ''), 1200) AS text
        FROM court_segments c
        LEFT JOIN court_considerations t USING (citation)
        WHERE c.citation = '{citation.replace("'", "''")}'
        LIMIT 1
        '''
    ).fetchone()
    if row:
        return f"Citation: {row[0]}\nType: court\nPattern: {row[1]}\nSubfamily: {row[2]}\nDecision: {row[3]}\nPrefix: {row[4]}\nBGE division: {row[5]}\nYear: {row[6]}\nText: {row[7]}"
    return f"Citation: {citation}"


def rerank_candidates_for_query(query: str, candidates: list[str], limit: int = 500) -> pd.DataFrame:
    # Placeholder scoring interface. For very large pools, first use the funnel score or a smaller reranker batch.
    cards = [candidate_evidence_card(c) for c in candidates[:limit]]
    return pd.DataFrame({"citation": candidates[:limit], "evidence": cards})


if USE_RERANKER:
    load_reranker_model()
    print("Reranker loaded:", RERANKER_MODEL_NAME)
else:
    print("Reranker disabled. Set USE_RERANKER=True after candidate recall passes.")

Reranker disabled. Set USE_RERANKER=True after candidate recall passes.


## 14. Export Submission Candidate Diagnostics

For `RUN_SPLIT = "test"`, this writes candidate sets only. Final prediction selection should happen after reranking/threshold calibration.

In [19]:
if RUN_SPLIT == "test":
    test_diag = pd.DataFrame(
        [
            {
                "query_id": r["query_id"],
                "candidate_count": r["candidate_count"],
                "law_candidate_count": r["law_candidate_count"],
                "court_candidate_count": r["court_candidate_count"],
            }
            for r in candidate_rows
        ]
    )
    path = ART_DIR / "test_candidate_diagnostics.csv"
    test_diag.to_csv(path, index=False)
    display(test_diag)
    print("Saved:", path)